# Figure 2 — running example

Independent, reproducible model walkthrough. **All computed values are [ours-scaled-down].** The figure specifies only six directed arcs; no missing distances are invented.

Install the package with `pip install -e .[test]` from the repository root before running this notebook.

In [ ]:
from befrg import paper_example, BEFRGSolver, validate_solution
from befrg.physical import solve_physical_grid, validate_physical_result
from experiments.milp import continuous_milp
p = paper_example()
print("Battery (kWh):", p.battery, "charge levels:", p.gamma, "residual levels:", p.Gamma)

## Original floored-state DP
The original DP discards the fractional remainder at each fixed-route node. Its result is optimal only for this *discrete* model.

In [ ]:
base=BEFRGSolver(p).bottom_up()
physical=validate_solution(p,base)
print("[ours-scaled-down] DP cost", base.total_minutes, "min")
print("[ours-scaled-down] chosen path", base.full_path)
print("DP states", base.state_trace)
print("Actual feasible", physical.feasible, "physical time", physical.total_minutes)

In [ ]:
for i, visit in enumerate(physical.visits):
    print(f"{i:2} {visit.node:>6} {visit.event:>19} SoC={visit.soc_kwh:7.3f} kWh elapsed={visit.elapsed_min:8.3f} min")

## Physical-state extension and continuous MILP check
The corrected planner preserves physical SoC between customers and uses discrete station charge targets. On the sparse Figure 2 graph, the continuous direct route is feasible and does not require a detour.

In [ ]:
corrected=solve_physical_grid(p)
valid,errs=validate_physical_result(p,corrected)
milp=continuous_milp(p,time_limit=15.)
print("[ours-scaled-down] corrected", corrected.solution.total_minutes, "min; valid", valid, errs)
print("[ours-scaled-down] continuous MILP", milp.objective_min, "certified", milp.certified_optimal)
print("Actions", [(a.kind,a.charge_level) for a in corrected.solution.actions])

## Caveats
The real figure omits many station distances. Equality of the corrected finite-grid solution and the continuous MILP here is a property of the *sparse specified arcs*, not a general statement about all EV charging problems.